In [ ]:

from pyspark.sql.types import StringType, DoubleType
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import udf
import math
from pyspark.sql.window import Window


Member 1

In [ ]:
# ── Member 1 | Cell 1: Initialize Spark Session ──────────────────────────────
# This session object is reused by ALL members. Export it at the end of this
# section so Members 2–6 can attach to the same running session.

spark = SparkSession.builder \
    .appName("CitiBike_BigData_Project") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")  # reduce noise in output
print("Spark version:", spark.version)
spark.sparkContext.uiWebUrl          # print Spark UI link

Spark version: 4.0.2


'http://8bd56e3cafc9:4040'

In [ ]:
# ── Member 1 | Cell 2: Load CSV ───────────────────────────────────────────────
# inferSchema reads the file once to detect types.

raw_df = spark.read.csv(
    "citi_data.csv",
    header=True,
    inferSchema=True
)

print(f"Row count: {raw_df.count():,}")
print(f"Column count: {len(raw_df.columns)}")
raw_df.printSchema()

Row count: 1,300,000
Column count: 15
root
 |-- _c0: integer (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start station id: double (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station id: double (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: integer (nullable = true)
 |-- gender: integer (nullable = true)



In [ ]:
# ── Member 1 | Cell 3: Explore Dataset ───────────────────────────────────────
# Get a feel for the data before touching it. Show sample rows and basic stats.

raw_df.show(5, truncate=False)
raw_df.describe().show()  # numeric column stats (min, max, mean, stddev)

+---+-----------------------+-----------------------+----------------+------------------------+----------------------+-----------------------+--------------+-----------------------------+--------------------+---------------------+------+----------+----------+------+
|_c0|starttime              |stoptime               |start station id|start station name      |start station latitude|start station longitude|end station id|end station name             |end station latitude|end station longitude|bikeid|usertype  |birth year|gender|
+---+-----------------------+-----------------------+----------------+------------------------+----------------------+-----------------------+--------------+-----------------------------+--------------------+---------------------+------+----------+----------+------+
|0  |2019-04-17 14:37:03.844|2019-04-17 14:43:13.767|264.0           |Maiden Ln & Pearl St    |40.70706456           |-74.00731853           |330.0         |Reade St & Broadway          |40.71450451 

In [ ]:
# ── Member 1 | Cell 4: Count NULLs per Column ────────────────────────────────
# We use a list comprehension to count nulls in every column at once.

null_counts = raw_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in raw_df.columns
])
null_counts.show(vertical=True)

-RECORD 0----------------------
 _c0                     | 0   
 starttime               | 0   
 stoptime                | 0   
 start station id        | 33  
 start station name      | 33  
 start station latitude  | 0   
 start station longitude | 0   
 end station id          | 33  
 end station name        | 33  
 end station latitude    | 0   
 end station longitude   | 0   
 bikeid                  | 0   
 usertype                | 0   
 birth year              | 0   
 gender                  | 0   



In [ ]:
# ── Member 1 | Cell 5: Detect Duplicates ─────────────────────────────────────
# A full duplicate = every column identical. We also check for duplicate bike_id
# + starttime combinations which would indicate a data entry error.

total_rows = raw_df.count()
distinct_rows = raw_df.distinct().count()
print(f"Total rows     : {total_rows:,}")
print(f"Distinct rows  : {distinct_rows:,}")
print(f"Duplicate rows : {total_rows - distinct_rows:,}")

Total rows     : 1,300,000
Distinct rows  : 1,300,000
Duplicate rows : 0


In [ ]:
# ── Member 1 | Cell 6: Detect Empty Strings & Invalid Values ─────────────────
# From the schema: start/end station name are STRING → check empty strings.
# usertype is STRING → check empty strings.
# gender and birth_year are INTEGER (already inferred) → no empty string check.
# Coordinates are DOUBLE → check for zero values (impossible in NYC).

# String columns only
empty_check = raw_df.select([
    F.count(F.when(F.col(c) == "", c)).alias(c)
    for c in ["start station name", "end station name", "usertype"]
])
empty_check.show()

# Check for zero-valued coordinates
zero_coords = raw_df.filter(
    (F.col("start station latitude") == 0) |
    (F.col("start station longitude") == 0) |
    (F.col("end station latitude") == 0) |
    (F.col("end station longitude") == 0)
).count()
print(f"Rows with zero coordinates: {zero_coords}")

# Check gender for values outside {0, 1, 2}
invalid_gender = raw_df.filter(
    ~F.col("gender").isin(0, 1, 2)
).count()
print(f"Rows with invalid gender value: {invalid_gender}")

# Check birth_year range (dataset is 2019, so valid age 12–100 → year 1919–2007)
raw_df.select(
    F.min("birth year").alias("min_birth_year"),
    F.max("birth year").alias("max_birth_year")
).show()

+------------------+----------------+--------+
|start station name|end station name|usertype|
+------------------+----------------+--------+
|                 0|               0|       0|
+------------------+----------------+--------+

Rows with zero coordinates: 0
Rows with invalid gender value: 0
+--------------+--------------+
|min_birth_year|max_birth_year|
+--------------+--------------+
|          1874|          2003|
+--------------+--------------+



In [ ]:
# ── Member 1 | Cell 7: Deduplication ─────────────────────────────────────────
# Cell 5 confirmed 0 duplicate rows (1,300,000 total = 1,300,000 distinct).
# We still call dropDuplicates() for safety and to document the step.

df = raw_df.dropDuplicates()
print(f"After dedup: {df.count():,} rows (no change expected)")

After dedup: 1,300,000 rows (no change expected)


In [ ]:
# ── Member 1 | Cell 8: Drop Rows Missing Essential Identifiers ───────────────
# Using ACTUAL column names from the schema (with spaces, e.g. "start station name").
# Cell 4 showed 0 NULLs across all columns, so this is a safety net.
# We also drop rows where usertype is empty (unusable for analysis).

df = df.filter(
    F.col("start station name").isNotNull() & (F.col("start station name") != "") &
    F.col("end station name").isNotNull()   & (F.col("end station name") != "") &
    F.col("bikeid").isNotNull()             &
    F.col("usertype").isNotNull()           & (F.col("usertype") != "")
)
print(f"After essential-identifier cleaning: {df.count():,} rows")

After essential-identifier cleaning: 1,299,967 rows


In [ ]:
# ── Member 1 | Cell 9: Remove Zero Coordinates ───────────────────────────────
# Actual column names: "start station latitude", "start station longitude",
# "end station latitude", "end station longitude" (with spaces, no underscores).

df = df.filter(
    (F.col("start station latitude") != 0)  &
    (F.col("start station longitude") != 0) &
    (F.col("end station latitude") != 0)    &
    (F.col("end station longitude") != 0)
)
print(f"After zero-coordinate cleaning: {df.count():,} rows")

After zero-coordinate cleaning: 1,299,967 rows


In [ ]:
# ── Member 1 | Cell 10: Type Validation ──────────────────────────────────────
# Cell 2 schema shows starttime/stoptime already loaded as TIMESTAMP ✓
# birth year already loaded as INTEGER ✓
# No casting needed. We just validate and document.

# Confirm no timestamps are NULL or illogical (stoptime before starttime)
bad_times = df.filter(
    F.col("starttime").isNull() |
    F.col("stoptime").isNull()  |
    (F.col("stoptime") <= F.col("starttime"))
).count()
print(f"Rows with bad/missing timestamps: {bad_times}")

# Confirm birth year has no NULLs (we saw 0 NULLs earlier, but verify on df)
null_birth = df.filter(F.col("birth year").isNull()).count()
print(f"Rows with NULL birth year: {null_birth}")

df.printSchema()

Rows with bad/missing timestamps: 0
Rows with NULL birth year: 0
root
 |-- _c0: integer (nullable = true)
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start station id: double (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station id: double (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: integer (nullable = true)
 |-- gender: integer (nullable = true)



In [ ]:
# ── Member 1 | Cell 11: Noise Flagging ───────────────────────────────────────
# Dataset year is 2019 (seen in starttime: 2019-04-17).
# Noise criteria from the project spec:
#   1. Trip duration < 60 seconds
#   2. Rider age > 100 or < 12  (birth year < 1919 or > 2007 for year 2019)
#   3. Missing essential identifiers (belt-and-suspenders)
# Criterion 4 (speed > 40 km/h) is added by Member 2 after computing distance.

DATASET_YEAR = 2019

df = df.withColumn(
    "noise_flag",
    F.when(
        # Criterion 1: trip under 60 seconds
        (F.unix_timestamp("stoptime") - F.unix_timestamp("starttime") < 60) |
        # Criterion 2: impossible birth year
        (
            F.col("birth year").isNotNull() &
            (
                ((DATASET_YEAR - F.col("birth year")) > 100) |
                ((DATASET_YEAR - F.col("birth year")) < 12)
            )
        ) |
        # Criterion 3: missing essentials (catches anything slipping through)
        F.col("start station name").isNull() |
        F.col("end station name").isNull()   |
        F.col("bikeid").isNull(),
        1
    ).otherwise(0)
)

print("Noise flag distribution:")
df.groupBy("noise_flag").count().orderBy("noise_flag").show()

Noise flag distribution:
+----------+-------+
|noise_flag|  count|
+----------+-------+
|         0|1299313|
|         1|    654|
+----------+-------+



In [ ]:
# ── Member 1 | Cell 12: Final Validation ─────────────────────────────────────

print("=== Final cleaned_df summary ===")
total      = df.count()
noisy      = df.filter(F.col("noise_flag") == 1).count()
clean      = df.filter(F.col("noise_flag") == 0).count()
print(f"Total rows  : {total:,}")
print(f"Noise rows  : {noisy:,}")
print(f"Clean rows  : {clean:,}")
df.show(5, truncate=False)

=== Final cleaned_df summary ===
Total rows  : 1,299,967
Noise rows  : 654
Clean rows  : 1,299,313
+---+-----------------------+-----------------------+----------------+-----------------------------+----------------------+-----------------------+--------------+---------------------------+--------------------+---------------------+------+----------+----------+------+----------+
|_c0|starttime              |stoptime               |start station id|start station name           |start station latitude|start station longitude|end station id|end station name           |end station latitude|end station longitude|bikeid|usertype  |birth year|gender|noise_flag|
+---+-----------------------+-----------------------+----------------+-----------------------------+----------------------+-----------------------+--------------+---------------------------+--------------------+---------------------+------+----------+----------+------+----------+
|336|2019-04-17 14:43:00.002|2019-04-17 14:57:53.878|3167.

In [ ]:
# ── Member 1 | Cell 12.5: Drop Redundant & Useless Columns ───────────────────
# After reviewing the schema we identified 3 columns to remove:
#
#   _c0              → auto-generated row index from CSV, carries no information
#   start station id → redundant: station is already fully identified by
#                      "start station name" + lat/long coordinates
#   end station id   → same reason as above
#
# Keeping station names and coordinates because:
#   - Names are used in queries (popular stations, station pairs)
#   - Coordinates are used by Member 2 for Haversine distance calculation
# Station IDs are numeric surrogates that offer nothing extra once we have both.

df = df.drop("_c0", "start station id", "end station id")

print("Remaining columns:")
df.printSchema()
print(f"Column count after dropping redundant cols: {len(df.columns)}")

Remaining columns:
root
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: integer (nullable = true)
 |-- gender: integer (nullable = true)
 |-- noise_flag: integer (nullable = false)

Column count after dropping redundant cols: 13


In [ ]:
# ── Member 1 | Cell 13: Save Final Cleaned DataFrame ─────────────────────────

df.write.mode("overwrite").parquet("cleaned_citi_data.parquet")
print("Saved to cleaned_citi_data.parquet ✓")
print(f"Final shape: {df.count():,} rows × {len(df.columns)} columns")

# ── NOTE FOR ALL MEMBERS ──────────────────────────────────────────────────────
# Load with:
#   cleaned_df = spark.read.parquet("cleaned_citi_data.parquet")
#   clean_only = cleaned_df.filter(F.col("noise_flag") == 0)
#
# Final columns (13 total):
#   starttime, stoptime,
#   start station name, start station latitude, start station longitude,
#   end station name,   end station latitude,   end station longitude,
#   bikeid, usertype, birth year, gender, noise_flag
#
# Column names have SPACES — always quote: F.col("start station name")
# Member 2: add speed_noise_flag after computing trip_speed

Saved to cleaned_citi_data.parquet ✓
Final shape: 1,299,967 rows × 13 columns


# 🚲 Member 1 — Data Cleaning & Validation Summary
**Project:** Citi Bike Trip Analysis | Big Data & NoSQL, Spring 2026
**Dataset:** `citi_data.csv` → **Output:** `cleaned_citi_data.parquet`

---

## Dataset Overview

| Property | Raw | After Cleaning |
|---|---|---|
| Rows | 1,300,000 | 1,299,967 |
| Columns | 15 | 13 |
| Noisy rows (flagged) | — | 654 |
| Clean rows (noise_flag = 0) | — | 1,299,313 |
| Date range | April 2019 | April 2019 |

---

## Schema & Column Decisions

| Column | Type | Decision | Reason |
|---|---|---|---|
| `_c0` | Integer | ❌ Dropped | Auto-generated CSV index, zero analytical value |
| `starttime` | Timestamp | ✅ Kept | Trip start time — used for duration, hour, month, weekday |
| `stoptime` | Timestamp | ✅ Kept | Trip end time — used for duration calculation |
| `start station id` | Double | ❌ Dropped | Redundant — station fully identified by name + coordinates |
| `start station name` | String | ✅ Kept | Primary station identifier for all queries |
| `start station latitude` | Double | ✅ Kept | Required for Haversine distance (Member 2) |
| `start station longitude` | Double | ✅ Kept | Required for Haversine distance (Member 2) |
| `end station id` | Double | ❌ Dropped | Redundant — same reason as start station id |
| `end station name` | String | ✅ Kept | Used in destination analysis and station pair queries |
| `end station latitude` | Double | ✅ Kept | Required for Haversine distance (Member 2) |
| `end station longitude` | Double | ✅ Kept | Required for Haversine distance (Member 2) |
| `bikeid` | Integer | ✅ Kept | Essential — used for bike utilization analysis |
| `usertype` | String | ✅ Kept | Subscriber vs Customer — used across all queries |
| `birth year` | Integer | ✅ Kept | Used to derive age and age group |
| `gender` | Integer | ✅ Kept | 0=Unknown, 1=Male, 2=Female — ML prediction target |
| `noise_flag` | Integer | ✅ Added | 0=clean, 1=noisy — lets each member filter as needed |

---

## Why We Dropped `_c0`

`_c0` is an auto-generated row index that Apache Spark creates when reading a CSV file that has an unnamed first column. It carries no information about the trip itself — it is simply 0, 1, 2, 3... up to 1,299,999. Keeping it would:

- Waste memory and storage across 1.3M rows
- Risk confusing downstream members who might mistake it for a meaningful identifier
- Add no value to any query, feature, or ML model

**Decision: Drop `_c0` immediately. It is metadata about the file, not data about the trips.**

---

## Why We Kept Station Name and Dropped Station ID

Both `start station id` and `start station name` were present in the raw data, and the `describe()` output confirmed they have the **exact same row count (1,299,967)** — meaning they are perfectly 1-to-1 with no situation where one exists without the other.

**We kept station name and dropped station ID for the following reasons:**

**1. Every analytical query needs human-readable names.**
Queries b, g, and h — most popular stations, subscriber vs customer destination analysis, and top station pairs — all require station names in their output. An output showing `264.0 → 330.0` is meaningless. An output showing `Maiden Ln & Pearl St → Reade St & Broadway` is immediately interpretable.

**2. Station IDs are floating point surrogates with no lookup table.**
The IDs (`264.0`, `3556.0`, `3908.0`) are numeric keys that only make sense if you have a separate stations reference table — which we do not. Without it, the ID alone tells you nothing about location, neighborhood, or usage pattern.

**3. Distance calculation only needs coordinates.**
Member 2's Haversine formula uses lat/long exclusively. Station name and station ID are both irrelevant for that calculation — so keeping the ID for distance purposes would be wrong reasoning.

**4. The 33 NULL rows had both name AND ID as NULL simultaneously.**
Station ID would not have rescued those rows — both fields were missing together while coordinates remained valid. This confirms that name and ID carry identical information and fail under the same conditions.

**Conclusion: Station name + lat/long fully and unambiguously identifies every station. The ID is a redundant numeric surrogate that adds complexity without adding information.**

---

## Handling of NULL Station Name Rows

**Observation:** 33 rows had NULL values in `start station name`, `end station name`, `start station id`, and `end station id` — but their coordinates (lat/long) were fully valid. This means the system recorded *where* the trip happened physically, but not *which named station* it belonged to.

**Decision: Drop these 33 rows.**

**Reasoning:**
- 33 rows represent only **0.0025%** of the 1.3M dataset — dropping them is statistically negligible and has zero impact on any analysis or model.
- Station name is a **required field** for multiple analytical queries (most popular stations, destination analysis, station pairs). These rows could not participate in those queries meaningfully.
- Imputation was considered but rejected: reverse-geocoding or cross-referencing coordinates to guess a station name introduces risk — if the coordinates are even slightly off, we would silently assign the wrong name, which is worse than omitting the row entirely.
- **Dropping bad data is always preferable to fabricating uncertain data.**

---

## Data Quality Findings

| Check | Result |
|---|---|
| NULL values | 33 rows had NULL station name + station id (coordinates were valid) |
| Duplicates | 0 duplicate rows found |
| Empty strings (station name, usertype) | 0 |
| Zero coordinates | 0 |
| Invalid gender values (outside 0/1/2) | 0 |
| Bad/inverted timestamps | 0 |
| NULL birth year | 0 |
| Birth year range | 1874 – 2003 (min age = 16, max age = 145 → noise flagged) |

---

## Cleaning Steps

| Step | Cell | Action | Rows Before | Rows After | Dropped |
|---|---|---|---|---|---|
| Load raw CSV | 2 | `spark.read.csv` with inferSchema | — | 1,300,000 | — |
| Explore & describe | 3 | `show()` + `describe()` | 1,300,000 | — | — |
| NULL check | 4 | Count nulls per column | 1,300,000 | — | — |
| Duplicate check | 5 | `distinct().count()` | 1,300,000 | — | — |
| Empty string / invalid check | 6 | Check strings, coords, gender, birth year | 1,300,000 | — | — |
| Drop duplicates | 7 | `dropDuplicates()` | 1,300,000 | 1,300,000 | 0 |
| Drop NULL identifiers | 8 | Filter NULL/empty station name & bikeid | 1,300,000 | 1,299,967 | **33** |
| Drop zero coordinates | 9 | Filter lat/long == 0 | 1,299,967 | 1,299,967 | 0 |
| Timestamp validation | 10 | Check NULL/inverted timestamps | 1,299,967 | 1,299,967 | 0 |
| Noise flagging | 11 | Add `noise_flag` column | 1,299,967 | 1,299,967 | 654 flagged |
| Final validation | 12 | Count total / clean / noisy rows | 1,299,967 | — | — |
| Drop redundant columns | 12.5 | Remove `_c0`, `start/end station id` | 15 cols | 13 cols | — |
| Save to parquet | 13 | `df.write.parquet(...)` | 1,299,967 | 1,299,967 | — |

---

## Noise Flagging Criteria

| # | Criterion | Threshold |
|---|---|---|
| 1 | Trip duration too short | < 60 seconds |
| 2 | Rider age impossible | > 100 or < 12 years (birth year < 1919 or > 2007) |
| 3 | Missing essential identifiers | NULL station name or bikeid (belt-and-suspenders) |
| 4 | Trip speed too high | > 40 km/h ⚠️ **Deferred to Member 2** (needs Haversine distance first) |

**Total flagged: 654 rows (0.05%) → 1,299,313 fully clean rows (99.95%)**

> Note: Noise rows are **flagged, not dropped**, so each member can decide whether to include or exclude them depending on their specific analysis.

---


## Handoff to Other Members

| Member | Loads | Key instructions |
|---|---|---|
| **Member 2** | `cleaned_citi_data.parquet` | Use all rows; compute `trip_duration`, `trip_distance`, `trip_speed`; add `speed_noise_flag` for speed > 40 km/h |
| **Member 3** | Member 2's `feature_df` | Filter `noise_flag == 0` before queries |
| **Member 4** | Member 2's `feature_df` | Filter `noise_flag == 0` before queries |
| **Member 5** | Member 2's `feature_df` | Filter `noise_flag == 0` **AND** `gender != 0` (unknown gender unusable as ML label) |
| **Member 6** | All query outputs | Uses aggregated results, not raw data |

> ⚠️ **All column names contain spaces** — always use: `F.col("start station name")` not `F.col("start_station_name")`

Member2

In [ ]:
# ── Member 2 | Cell 1: Load Cleaned Data ─────────────────────────────

# Load cleaned parquet from Member 1
cleaned_df = spark.read.parquet("cleaned_citi_data.parquet")

print(f"Rows loaded: {cleaned_df.count():,}")
cleaned_df.printSchema()

Rows loaded: 1,299,967
root
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: integer (nullable = true)
 |-- gender: integer (nullable = true)
 |-- noise_flag: integer (nullable = true)



In [ ]:
# ── Member 2 | Cell 2: Rider Age Feature ────────────────────────────

DATASET_YEAR = 2019

feature_df = cleaned_df.withColumn(
    "rider_age",
    DATASET_YEAR - F.col("birth year")
)

feature_df.select(
    "birth year",
    "rider_age"
).show(5)

+----------+---------+
|birth year|rider_age|
+----------+---------+
|      1984|       35|
|      1997|       22|
|      1996|       23|
|      1962|       57|
|      1968|       51|
+----------+---------+
only showing top 5 rows


rider_age is calculated as 2019 - birth year

In [ ]:
# ── Member 2 | Cell 3: Trip Duration Feature ────────────────────────

feature_df = feature_df.withColumn(
    "trip_duration_sec",
    F.unix_timestamp("stoptime") - F.unix_timestamp("starttime")
)

feature_df.select(
    "starttime",
    "stoptime",
    "trip_duration_sec"
).show(5, truncate=False)

+-----------------------+-----------------------+-----------------+
|starttime              |stoptime               |trip_duration_sec|
+-----------------------+-----------------------+-----------------+
|2019-04-17 14:37:10.255|2019-04-17 14:42:26.559|316              |
|2019-04-17 14:37:34.609|2019-04-17 14:58:36.402|1262             |
|2019-04-17 14:38:36.756|2019-04-17 15:03:56.047|1520             |
|2019-04-17 14:40:07.935|2019-04-17 14:53:56.735|829              |
|2019-04-17 14:46:19.496|2019-04-17 14:54:14.25 |475              |
+-----------------------+-----------------------+-----------------+
only showing top 5 rows


In [ ]:
# ── Member 2 | Cell 4: Trip Distance (Haversine Formula) ────────────

EARTH_RADIUS_KM = 6371.0

feature_df = feature_df.withColumn(
    "lat1", F.radians(F.col("start station latitude"))
).withColumn(
    "lon1", F.radians(F.col("start station longitude"))
).withColumn(
    "lat2", F.radians(F.col("end station latitude"))
).withColumn(
    "lon2", F.radians(F.col("end station longitude"))
)

feature_df = feature_df.withColumn(
    "dlat",
    F.col("lat2") - F.col("lat1")
).withColumn(
    "dlon",
    F.col("lon2") - F.col("lon1")
)

feature_df = feature_df.withColumn(
    "a",
    F.pow(F.sin(F.col("dlat") / 2), 2) +
    F.cos(F.col("lat1")) *
    F.cos(F.col("lat2")) *
    F.pow(F.sin(F.col("dlon") / 2), 2)
)

feature_df = feature_df.withColumn(
    "c",
    2 * F.atan2(
        F.sqrt(F.col("a")),
        F.sqrt(1 - F.col("a"))
    )
)

feature_df = feature_df.withColumn(
    "trip_distance_km",
    EARTH_RADIUS_KM * F.col("c")
)

feature_df.select(
    "trip_distance_km"
).show(5)

+------------------+
|  trip_distance_km|
+------------------+
|0.9347153634252897|
|  2.35847219112615|
| 3.820987613561039|
|2.5182517465979535|
|1.1930358006672939|
+------------------+
only showing top 5 rows


In [ ]:
# ── Member 2 | Cell 5: Trip Speed Feature ───────────────────────────

feature_df = feature_df.withColumn(
    "trip_duration_hr",
    F.col("trip_duration_sec") / 3600
)

feature_df = feature_df.withColumn(
    "trip_speed_kmh",
    F.when(
        F.col("trip_duration_hr") > 0,
        F.col("trip_distance_km") / F.col("trip_duration_hr")
    ).otherwise(None)
)

feature_df.select(
    "trip_distance_km",
    "trip_duration_sec",
    "trip_speed_kmh"
).show(5)

+------------------+-----------------+------------------+
|  trip_distance_km|trip_duration_sec|    trip_speed_kmh|
+------------------+-----------------+------------------+
|0.9347153634252897|              316|10.648656039022288|
|  2.35847219112615|             1262| 6.727812906540523|
| 3.820987613561039|             1520| 9.049707505802461|
|2.5182517465979535|              829| 10.93571325422513|
|1.1930358006672939|              475|  9.04195554189949|
+------------------+-----------------+------------------+
only showing top 5 rows


In [ ]:
# ── Member 2 | Cell 6: Period of Day Feature ────────────────────────

feature_df = feature_df.withColumn(
    "hour",
    F.hour("starttime")
)

feature_df = feature_df.withColumn(
    "period_of_day",
    F.when((F.col("hour") >= 5) & (F.col("hour") < 12), "Morning")
     .when((F.col("hour") >= 12) & (F.col("hour") < 17), "Afternoon")
     .when((F.col("hour") >= 17) & (F.col("hour") < 21), "Evening")
     .otherwise("Night")
)

feature_df.select(
    "starttime",
    "hour",
    "period_of_day"
).show(10, truncate=False)

+-----------------------+----+-------------+
|starttime              |hour|period_of_day|
+-----------------------+----+-------------+
|2019-04-17 14:37:10.255|14  |Afternoon    |
|2019-04-17 14:37:34.609|14  |Afternoon    |
|2019-04-17 14:38:36.756|14  |Afternoon    |
|2019-04-17 14:40:07.935|14  |Afternoon    |
|2019-04-17 14:46:19.496|14  |Afternoon    |
|2019-04-17 14:58:35.126|14  |Afternoon    |
|2019-04-17 15:07:02.574|15  |Afternoon    |
|2019-04-17 15:08:16.029|15  |Afternoon    |
|2019-04-17 15:09:08.923|15  |Afternoon    |
|2019-04-17 15:09:10.097|15  |Afternoon    |
+-----------------------+----+-------------+
only showing top 10 rows


In [ ]:
# ── Member 2 | Cell 7: Start Month Feature ──────────────────────────

feature_df = feature_df.withColumn(
    "start_month",
    F.month("starttime")
)

feature_df.select(
    "starttime",
    "start_month"
).show(10)

+--------------------+-----------+
|           starttime|start_month|
+--------------------+-----------+
|2019-04-17 14:37:...|          4|
|2019-04-17 14:37:...|          4|
|2019-04-17 14:38:...|          4|
|2019-04-17 14:40:...|          4|
|2019-04-17 14:46:...|          4|
|2019-04-17 14:58:...|          4|
|2019-04-17 15:07:...|          4|
|2019-04-17 15:08:...|          4|
|2019-04-17 15:09:...|          4|
|2019-04-17 15:09:...|          4|
+--------------------+-----------+
only showing top 10 rows


In [ ]:
# ── Member 2 | Cell 8: Age Group UDF ────────────────────────────────

def classify_age(age):
    if age is None:
        return "Unknown"
    elif age < 25:
        return "Young"
    elif age < 60:
        return "Adult"
    else:
        return "Senior"

age_group_udf = udf(classify_age, StringType())

feature_df = feature_df.withColumn(
    "age_group",
    age_group_udf(F.col("rider_age"))
)

feature_df.select(
    "rider_age",
    "age_group"
).show(10)

+---------+---------+
|rider_age|age_group|
+---------+---------+
|       35|    Adult|
|       22|    Young|
|       23|    Young|
|       57|    Adult|
|       51|    Adult|
|       20|    Young|
|       50|    Adult|
|       44|    Adult|
|       29|    Adult|
|       58|    Adult|
+---------+---------+
only showing top 10 rows


In [ ]:
# ── Member 2 | Cell 9: Season UDF ───────────────────────────────────

def classify_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

season_udf = udf(classify_season, StringType())

feature_df = feature_df.withColumn(
    "season",
    season_udf(F.col("start_month"))
)

feature_df.select(
    "start_month",
    "season"
).show(10)

+-----------+------+
|start_month|season|
+-----------+------+
|          4|Spring|
|          4|Spring|
|          4|Spring|
|          4|Spring|
|          4|Spring|
|          4|Spring|
|          4|Spring|
|          4|Spring|
|          4|Spring|
|          4|Spring|
+-----------+------+
only showing top 10 rows


In [ ]:
# ── Member 2 | Cell 10: Speed Noise Flag ────────────────────────────

feature_df = feature_df.withColumn(
    "speed_noise_flag",
    F.when(F.col("trip_speed_kmh") > 40, 1).otherwise(0)
)

feature_df.groupBy("speed_noise_flag").count().show()

+----------------+-------+
|speed_noise_flag|  count|
+----------------+-------+
|               1|      1|
|               0|1299966|
+----------------+-------+



In [ ]:
# ── Member 2 | Cell 11: Feature Validation ──────────────────────────

print("=== Feature Validation Summary ===")

feature_df.select(
    F.min("rider_age").alias("min_age"),
    F.max("rider_age").alias("max_age"),

    F.min("trip_duration_sec").alias("min_duration"),
    F.max("trip_duration_sec").alias("max_duration"),

    F.min("trip_distance_km").alias("min_distance"),
    F.max("trip_distance_km").alias("max_distance"),

    F.min("trip_speed_kmh").alias("min_speed"),
    F.max("trip_speed_kmh").alias("max_speed")
).show()

print("Period of day distribution:")
feature_df.groupBy("period_of_day").count().show()

print("Age group distribution:")
feature_df.groupBy("age_group").count().show()

print("Season distribution:")
feature_df.groupBy("season").count().show()

=== Feature Validation Summary ===
+-------+-------+------------+------------+------------+------------------+---------+------------------+
|min_age|max_age|min_duration|max_duration|min_distance|      max_distance|min_speed|         max_speed|
+-------+-------+------------+------------+------------+------------------+---------+------------------+
|     16|    145|          61|     2943038|         0.0|16.732761099958836|      0.0|52.969812151630336|
+-------+-------+------------+------------+------------+------------------+---------+------------------+

Period of day distribution:
+-------------+------+
|period_of_day| count|
+-------------+------+
|      Evening|336966|
|      Morning|433531|
|    Afternoon|438090|
|        Night| 91380|
+-------------+------+

Age group distribution:
+---------+-------+
|age_group|  count|
+---------+-------+
|   Senior|  74442|
|    Adult|1119429|
|    Young| 106096|
+---------+-------+

Season distribution:
+------+------+
|season| count|
+------+

> **Note:** The maximum age of 145 includes noisy rows (`noise_flag=1`). Clean rows (used in all queries and ML) have rider age ≤ 100, as per project specification.

In [ ]:
# ── Member 2 | Cell 12: Drop Temporary Columns ──────────────────────

feature_df = feature_df.drop(
    "lat1", "lon1",
    "lat2", "lon2",
    "dlat", "dlon",
    "a", "c",
    "trip_duration_hr",
    "hour"
)

print(f"Final column count: {len(feature_df.columns)}")
feature_df.printSchema()

Final column count: 22
root
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: integer (nullable = true)
 |-- gender: integer (nullable = true)
 |-- noise_flag: integer (nullable = true)
 |-- rider_age: integer (nullable = true)
 |-- trip_duration_sec: long (nullable = true)
 |-- trip_distance_km: double (nullable = true)
 |-- trip_speed_kmh: double (nullable = true)
 |-- period_of_day: string (nullable = false)
 |-- start_month: integer (nullable = true)
 |-- age_group: string (nullable = true)
 |-- season: string (nullable = true)
 |-- speed_noise_

In [ ]:
# ── Member 2 | Cell 13: Save Feature DataFrame ──────────────────────

feature_df.write.mode("overwrite").parquet("feature_engineered_citi_data.parquet")

print("Saved feature_engineered_citi_data.parquet ✓")

print(f"Final rows   : {feature_df.count():,}")
print(f"Final columns: {len(feature_df.columns)}")

Saved feature_engineered_citi_data.parquet ✓
Final rows   : 1,299,967
Final columns: 22


#  Member 2 — Feature Engineering & UDFs Summary

**Project:** Citi Bike Trip Analysis | Big Data & NoSQL, Spring 2026
**Input:** `cleaned_citi_data.parquet`
**Output:** `feature_engineered_citi_data.parquet`

---

## Objective of This Section

After Member 1 completed data cleaning and validation, the dataset contained reliable trip-level information but still lacked higher-level analytical features.

The purpose of Member 2’s work is to transform raw columns into meaningful engineered features that can later support:

- analytical queries
- behavioral analysis
- statistical comparisons
- visualization/dashboard creation
- machine learning models

This stage is called **feature engineering**.

Feature engineering converts raw data into informative variables that better describe user behavior, trip patterns, and operational characteristics.

The engineered dataset becomes the primary input for:

- Members 3 & 4 → analytical insights
- Member 5 → machine learning models
- Member 6 → visualizations/dashboard

---

## Input Dataset

Loaded from:

`cleaned_citi_data.parquet`

Generated by Member 1 after:

- NULL handling
- duplicate checking
- timestamp validation
- essential identifier filtering
- noise flagging

Input schema contained:

| Column                  | Type        |
|-------------------------|-------------|
| `starttime`               | Timestamp   |
| `stoptime`                | Timestamp   |
| `start station name`      | String      |
| `start station latitude`  | Double      |
| `start station longitude` | Double      |
| `end station name`        | String      |
| `end station latitude`    | Double      |
| `end station longitude`   | Double      |
| `bikeid`                  | Integer     |
| `usertype`                | String      |
| `birth year`              | Integer     |
| `gender`                  | Integer     |
| `noise_flag`              | Integer     |

Total rows loaded:

1,299,967 rows

---

## Feature Engineering Overview

The following new features were created:

| Feature          | Type        | Purpose                            |
|------------------|-------------|------------------------------------|
| `rider_age`          | Numeric     | Rider demographic analysis         |
| `trip_duration_sec`  | Numeric     | Trip length analysis               |
| `trip_distance_km`   | Numeric     | Geographic movement analysis       |
| `trip_speed_kmh`     | Numeric     | Behavioral and anomaly analysis    |
| `period_of_day`      | Categorical | Demand/rush-hour analysis          |
| `start_month`        | Numeric     | Seasonal analysis                  |
| `age_group`          | Categorical | Group-based rider analysis         |
| `season`             | Categorical | Seasonal grouping                  |
| `speed_noise_flag`   | Binary      | High-speed anomaly detection       |

---

### Rider Age Feature

**Objective**

Convert birth year into an interpretable rider age.

Raw birth year values are less meaningful analytically than actual rider age.

**Formula Used**

Dataset year was identified as 2019 from trip timestamps.

Therefore:

`Rider Age = 2019 - Birth Year`

**Example**

| Birth Year | Rider Age |
|------------|-----------|
| 1995       | 24        |
| 1980       | 39        |
| 1960       | 59        |

**Why 2019 Was Used Instead of Current Year**

The dataset records trips that occurred during April 2019.

Age must therefore represent the rider’s age at the time the trip occurred, not the rider’s current age in 2026.

Using 2026 would distort:

- age-group analysis
- rider behavior interpretation
- machine learning features

**Example:**

| Birth Year | Age in 2019 | Age in 2026 |
|------------|-------------|-------------|
| 1995       | 24          | 31          |

The rider behaved as a 24-year-old during the trip, not as a 31-year-old.

---

### Trip Duration Feature

**Objective**

Calculate total trip duration in seconds.

The dataset stores:

- `starttime`
- `stoptime`

but not trip duration directly.

**Formula Used**

`Trip Duration = Stop Time - Start Time`

Spark converts timestamps into Unix seconds before subtraction.

**Example**

| Start Time | Stop Time | Duration |
|------------|-----------|----------|
| 10:00:00   | 10:15:00  | 900 sec  |

**Why Seconds Were Used**

Seconds are preferred because they:

- provide precise measurements
- avoid rounding errors
- simplify ML preprocessing
- support accurate speed calculation

---

### Trip Distance Feature (Haversine Formula)

**Objective**

Estimate the geographic distance traveled during each trip.

The dataset contains:

- start coordinates
- end coordinates

but not actual trip distance.

**Why Euclidean Distance Was Rejected**

Latitude and longitude exist on a curved Earth surface.

Straight-line Euclidean distance would introduce geographic inaccuracies.

Instead, the **Haversine formula** was used because it calculates great-circle distance over Earth’s surface.

**Haversine Formula**

```
d = 2 * r * arcsin(sqrt(sin^2(Δφ/2) + cos(φ1) * cos(φ2) * sin^2(Δλ/2)))
```

Where:

| Symbol | Meaning           |
|--------|-------------------|
| `r`      | Earth radius (6371 km) |
| `φ`      | Latitude          |
| `λ`      | Longitude         |
| `Δφ`     | Latitude difference |
| `Δλ`     | Longitude difference |

**Haversine Calculation Steps**

1. Convert latitude/longitude from degrees → radians
2. Compute latitude and longitude differences
3. Compute intermediate angular separation values
4. Convert angular separation into curved Earth distance
5. Multiply by Earth radius

Earth radius used:

6371 km

**Why This Feature Is Important**

Trip distance is foundational for:

- trip speed calculation
- station demand analysis
- rider behavior analysis
- bike utilization analysis
- machine learning features

Without geographic distance, later members cannot measure realistic trip movement.

---

### Trip Speed Feature

**Objective**

Calculate rider speed in kilometers per hour.

**Formula Used**

`v = d / t`

Where:

- `d` = trip distance (km)
- `t` = trip duration (hours)

**Unit Conversion**

Trip duration was initially measured in seconds.

Conversion:

`Hours = Seconds / 3600`

**Example**

| Distance | Duration | Speed    |
|----------|----------|----------|
| 5 km     | 0.5 hr   | 10 km/h  |

**Protection Against Division by Zero**

Trips with zero duration could cause invalid division operations.

Spark’s `when()` logic was used to safely avoid division by zero.

---

### Period of Day Feature

**Objective**

Group trips into human-readable daily activity periods.

This simplifies demand analysis and rush-hour detection.

**Categorization Logic**

| Hour Range | Category  |
|------------|-----------|
| 5–11       | Morning   |
| 12–16      | Afternoon |
| 17–20      | Evening   |
| Otherwise  | Night     |

**Example**

| Start Time | Hour | Period    |
|------------|------|-----------|
| 08:15      | 8    | Morning   |
| 14:30      | 14   | Afternoon |
| 18:45      | 18   | Evening   |

**Why This Feature Matters**

Used later for:

- rush-hour analysis
- commuting behavior analysis
- temporal demand visualization
- hourly usage patterns

---

### Start Month Feature

**Objective**

Extract month number from trip timestamps.

**Example**

| Timestamp    | Month |
|--------------|-------|
| 2019-04-17   | 4     |

**Why This Feature Matters**

Needed for:

- seasonal analysis
- trend grouping
- season classification UDF

---

### Age Group UDF

**Objective**

Convert raw numeric ages into broader demographic groups.

Grouped demographics are easier to analyze than individual ages.

**What Is a UDF?**

`UDF = User Defined Function`

Spark provides many built-in functions, but custom grouping logic must be created manually using UDFs.

The UDF allows Spark to distribute the custom logic across millions of rows efficiently.

**Classification Logic**

| Age Range | Group   |
|-----------|---------|
| < 25      | Young   |
| 25–59     | Adult   |
| 60+       | Senior  |

**Example**

| Rider Age | Age Group |
|-----------|-----------|
| 21        | Young     |
| 40        | Adult     |
| 68        | Senior    |

**Why This Feature Matters**

Used for:

- duration comparisons
- demographic behavior analysis
- visualization grouping
- ML feature engineering

---

### Season Classification UDF

**Objective**

Convert numeric months into seasonal categories.

**Classification Logic**

| Months    | Season |
|-----------|--------|
| Dec–Feb   | Winter |
| Mar–May   | Spring |
| Jun–Aug   | Summer |
| Sep–Nov   | Autumn |

**Important Dataset Observation**

The dataset contains trips from multiple months across 2019, therefore all four seasons (Spring, Summer, Autumn, Winter) are represented in the `season` column. The UDF correctly classifies each trip based on its start month.
---

### Speed Noise Flag

**Objective**

Flag suspiciously high bike speeds.

**Threshold Selection**

Trips exceeding:

`40 km/h`

were marked as potential anomalies.

**Why 40 km/h Was Chosen**

Typical public bicycle speeds in urban environments usually range between:

- 10–25 km/h for normal riders
- 25–35 km/h for fast riders

Speeds above 40 km/h are uncommon due to:

- traffic lights
- intersections
- pedestrian activity
- road congestion

Very high calculated speeds often indicate:

- GPS inaccuracies
- timestamp inconsistencies
- coordinate mismatches
- corrupted records

**Important Design Decision**

Rows were **NOT deleted**.

Instead, a new binary feature was added:

`speed_noise_flag`
- `0` = normal
- `1` = suspicious

This preserves original data while still enabling optional filtering in later analyses.

---

### Temporary Haversine Columns Removal

Several intermediate columns were created during Haversine calculations:

| Temporary Column |
|------------------|
| `lat1`             |
| `lon1`             |
| `lat2`             |
| `lon2`             |
| `dlat`             |
| `dlon`             |
| `a`                |
| `c`                |

These columns were removed after final distance calculation because they:

- consume unnecessary memory
- clutter the schema
- provide no analytical value
- may confuse downstream members

Only the final meaningful feature:

`trip_distance_km`

was retained.

---

### Feature Validation

Validation checks were performed after all feature engineering steps.

**Validated Metrics**

| Validation              | Purpose                                   |
|-------------------------|-------------------------------------------|
| Minimum/maximum rider age | Detect impossible values                  |
| Minimum/maximum duration  | Detect negative durations                 |
| Minimum/maximum distance  | Verify Haversine correctness              |
| Minimum/maximum speed     | Detect unrealistic speeds                 |
| Category distributions  | Verify UDF correctness                    |

---

## Final Engineered Dataset

Saved as:

`feature_engineered_citi_data.parquet`

This dataset becomes the primary input for:

- Member 3 analytical queries
- Member 4 analytical queries
- Member 5 machine learning pipeline
- Member 6 visualization/dashboard creation

---

## Final Notes for Team Members

| Member   | Usage Instructions                                                                  |
|----------|-------------------------------------------------------------------------------------|
| Member 3 | Filter `noise_flag == 0` before analysis                                            |
| Member 4 | Filter `noise_flag == 0` before analysis                                            |
| Member 5 | Filter `noise_flag == 0` **AND** `gender != 0` (unknown gender unusable as ML label) |
| Member 6 | Use aggregated outputs for dashboard                                                |

> ⚠️ **All column names contain spaces** — always use: `F.col("start station name")` not `F.col("start_station_name")`

Member3

In [ ]:
# ── Member 3 | Cell 1: Load Data & Imports ─────────────────────────────

# Load the fully engineered dataset from Member 2
feature_df = spark.read.parquet("feature_engineered_citi_data.parquet")

print(f"Loaded {feature_df.count():,} rows")
feature_df.printSchema()

# For clean analysis we usually filter noise
clean_df = feature_df.filter(F.col("noise_flag") == 0)
print(f"Clean rows for analysis: {clean_df.count():,}")

Loaded 1,299,967 rows
root
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: integer (nullable = true)
 |-- gender: integer (nullable = true)
 |-- noise_flag: integer (nullable = true)
 |-- rider_age: integer (nullable = true)
 |-- trip_duration_sec: long (nullable = true)
 |-- trip_distance_km: double (nullable = true)
 |-- trip_speed_kmh: double (nullable = true)
 |-- period_of_day: string (nullable = true)
 |-- start_month: integer (nullable = true)
 |-- age_group: string (nullable = true)
 |-- season: string (nullable = true)
 |-- speed_noise_fl

In [ ]:
# ── Member 3 | Query a: Percentage of Round Trips per User Type ─────────

# Query (a): Round trips percentage per user type
round_trips_df = clean_df.withColumn(
    "is_round_trip",
    F.when(F.col("start station name") == F.col("end station name"), 1).otherwise(0)
)

result_a = round_trips_df.groupBy("usertype").agg(
    F.count("*").alias("total_trips"),
    F.sum("is_round_trip").alias("round_trips"),
    (F.round(F.sum("is_round_trip") / F.count("*") * 100, 2)).alias("round_trip_percentage")
).orderBy("usertype")

result_a.show(truncate=False)

+----------+-----------+-----------+---------------------+
|usertype  |total_trips|round_trips|round_trip_percentage|
+----------+-----------+-----------+---------------------+
|Customer  |183750     |9737       |5.3                  |
|Subscriber|1115563    |17960      |1.61                 |
+----------+-----------+-----------+---------------------+



Business Interpretation (a):

Subscribers have a significantly lower round-trip rate (1.61%) compared to Customers (5.30%). This confirms that subscribers primarily use Citi Bike for purposeful commuting, while Customers use it more for leisure and sightseeing.

For city planning, this highlights the need to provide better short-term parking solutions and docking capacity near tourist spots and recreational areas to accommodate casual users.

In [ ]:
# ── Member 3 | Query b: Most Popular Start Stations ─────────────────────

# Query (b): Most popular start stations overall and ranking
result_b = clean_df.groupBy("start station name").agg(
    F.count("*").alias("trip_count")
).orderBy(F.desc("trip_count"))

print("Top 15 Most Popular Start Stations:")
result_b.show(15, truncate=False)

Top 15 Most Popular Start Stations:
+-----------------------------+----------+
|start station name           |trip_count|
+-----------------------------+----------+
|Pershing Square North        |9761      |
|8 Ave & W 31 St              |8001      |
|E 17 St & Broadway           |7524      |
|Broadway & E 22 St           |7041      |
|W 21 St & 6 Ave              |7011      |
|Broadway & E 14 St           |6973      |
|West St & Chambers St        |6731      |
|Broadway & W 60 St           |6596      |
|Christopher St & Greenwich St|6557      |
|12 Ave & W 40 St             |6446      |
|W 20 St & 11 Ave             |6105      |
|Lafayette St & E 8 St        |5810      |
|W 41 St & 8 Ave              |5787      |
|8 Ave & W 33 St              |5758      |
|Broadway & W 25 St           |5612      |
+-----------------------------+----------+
only showing top 15 rows


Business Interpretation (b):

A small group of stations (e.g. Pershing Square North, 8 Ave & W 31 St) dominate usage.
These are likely key transit hubs and business districts.

City planners should focus on expanding capacity, increasing the number of available bikes, and improving maintenance at these high-demand stations to minimize shortages during peak periods.

In [ ]:
# ── Member 3 | Query c: Rush Hour Analysis ──────────────────────────────

# Query (c): Hours of the day with highest demand (Rush Hours)
result_c = clean_df.withColumn("hour_of_day", F.hour("starttime")) \
    .groupBy("hour_of_day") \
    .agg(F.count("*").alias("trip_count")) \
    .orderBy(F.desc("trip_count"))

print("Hourly Demand Distribution:")
result_c.show(24)

print("\nTop 5 Rush Hours:")
result_c.limit(5).show()

Hourly Demand Distribution:
+-----------+----------+
|hour_of_day|trip_count|
+-----------+----------+
|         17|    126578|
|          8|    115714|
|         18|    104873|
|         16|     98266|
|         15|     90935|
|         14|     88521|
|         13|     83817|
|          9|     79516|
|         12|     76324|
|          7|     70091|
|         11|     67774|
|         19|     64084|
|         10|     56660|
|         20|     41246|
|          6|     33068|
|         21|     27942|
|         22|     19573|
|          0|     12647|
|         23|     11566|
|          5|     10495|
|          1|      7939|
|          2|      5015|
|          4|      3573|
|          3|      3096|
+-----------+----------+


Top 5 Rush Hours:
+-----------+----------+
|hour_of_day|trip_count|
+-----------+----------+
|         17|    126578|
|          8|    115714|
|         18|    104873|
|         16|     98266|
|         15|     90935|
+-----------+----------+



Business Interpretation (c):

Peak demand occurs at 8 AM and 5 PM (17:00), with additional high demand from 15:00 to 18:00. This clear commuting pattern shows Citi Bike is heavily used for work-related travel.

Operational recommendations include proactive bike rebalancing before morning and evening peaks and exploring incentives to spread demand to off-peak hours.

In [ ]:
# ── Member 3 | Query d: Trip Duration by Age Group ──────────────────────

# Query (d): Trip duration variation across age groups
result_d = clean_df.groupBy("age_group").agg(
    F.count("*").alias("trip_count"),
    F.round(F.avg("trip_duration_sec"), 2).alias("avg_duration_sec"),
    F.round(F.percentile_approx("trip_duration_sec", 0.5), 2).alias("median_duration_sec"),
    F.round(F.avg("trip_distance_km"), 3).alias("avg_distance_km"),
    F.round(F.avg("trip_speed_kmh"), 2).alias("avg_speed_kmh")
).orderBy("age_group")

result_d.show(truncate=False)

+---------+----------+----------------+-------------------+---------------+-------------+
|age_group|trip_count|avg_duration_sec|median_duration_sec|avg_distance_km|avg_speed_kmh|
+---------+----------+----------------+-------------------+---------------+-------------+
|Adult    |1119429   |958.47          |615                |1.791          |8.84         |
|Senior   |73788     |841.97          |585                |1.602          |8.19         |
|Young    |106096    |1062.12         |647                |1.719          |8.41         |
+---------+----------+----------------+-------------------+---------------+-------------+



Business Interpretation (d):

Adult riders have the highest trip count (1,119,429) with an average duration of 958 seconds. Young riders have the longest average duration (1062 seconds) and highest median duration (647 seconds). Seniors have the shortest average duration (842 seconds) and lowest average speed (8.19 km/h).

These insights can help tailor safety messages, bike design (comfort for seniors), and station accessibility improvements according to different user demographics.

In [ ]:
# ── Member 3 | Query e: Seasonal Bike Usage Analysis ────────────────────

# Query (e): Bike usage and trip behavior across seasons
result_e = clean_df.groupBy("season").agg(
    F.count("*").alias("trip_count"),
    F.round(F.avg("trip_duration_sec"), 2).alias("avg_duration_sec"),
    F.round(F.avg("trip_distance_km"), 3).alias("avg_distance_km"),
    F.round(F.avg("trip_speed_kmh"), 2).alias("avg_speed_kmh")
).orderBy("season")

result_e.show(truncate=False)

# Monthly view
result_e_month = clean_df.groupBy("start_month").agg(
    F.count("*").alias("trip_count")
).orderBy("start_month")

print("Monthly Trip Distribution:")
result_e_month.show()

+------+----------+----------------+---------------+-------------+
|season|trip_count|avg_duration_sec|avg_distance_km|avg_speed_kmh|
+------+----------+----------------+---------------+-------------+
|Autumn|399808    |958.75          |1.791          |8.69         |
|Spring|299827    |939.94          |1.761          |9.09         |
|Summer|449780    |1018.31         |1.851          |8.56         |
|Winter|149898    |831.22          |1.531          |8.93         |
+------+----------+----------------+---------------+-------------+

Monthly Trip Distribution:
+-----------+----------+
|start_month|trip_count|
+-----------+----------+
|          1|     49960|
|          2|     49966|
|          3|     99942|
|          4|     99944|
|          5|     99941|
|          6|    149932|
|          7|    149924|
|          8|    149924|
|          9|    149933|
|         10|    149928|
|         11|     99947|
|         12|     49972|
+-----------+----------+



Business Interpretation (e):

Summer has the highest number of trips (449,780), followed by Autumn (399,808), Spring (299,827), and Winter (149,898). Summer trips have the longest average duration (1018 seconds) and longest average distance (1.851 km). Winter trips are the shortest in duration (831 seconds) and distance (1.531 km). Spring has the highest average speed (9.09 km/h), while Summer has the lowest (8.56 km/h).

Seasonal patterns show increased bike usage in warmer months, with longer, slower leisure rides in Summer, and shorter, faster commuting in Spring. Winter sees reduced activity but still substantial usage.

This information can guide seasonal bike availability, maintenance scheduling, and targeted marketing campaigns (e.g., promoting winter riding with gear or discounts).

In [ ]:
# ── Member 3 | Cell: Save Results for Dashboard ─────────────────────────

# Save aggregated results for Member 6
result_a.write.mode("overwrite").parquet("query_a_roundtrips.parquet")
result_b.limit(20).write.mode("overwrite").parquet("query_b_top_stations.parquet")
result_c.write.mode("overwrite").parquet("query_c_hourly.parquet")
result_d.write.mode("overwrite").parquet("query_d_agegroup.parquet")
result_e.write.mode("overwrite").parquet("query_e_seasonal.parquet")

print("All Member 3 query results saved successfully for visualization.")

All Member 3 query results saved successfully for visualization.


# Member 4

In [ ]:
#reload from member 2
feature_df = spark.read.parquet("feature_engineered_citi_data.parquet")


#filter out noisy data
#noise_flag = 0  row is not flagged  (valid duration, age, and identifiers) - choose noise_flag=1rows meaning clean data
#speed_noise_flag = 0  row is not flagged (speed under 40 km/h) - same here
clean_df = feature_df.filter(
    (F.col("noise_flag") == 0) & (F.col("speed_noise_flag") == 0)
)

print(f"Total rows loaded : {feature_df.count():,}")
print(f"Clean rows: {clean_df.count():,}")
clean_df.printSchema()


Total rows loaded : 1,299,967
Clean rows: 1,299,312
root
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: integer (nullable = true)
 |-- gender: integer (nullable = true)
 |-- noise_flag: integer (nullable = true)
 |-- rider_age: integer (nullable = true)
 |-- trip_duration_sec: long (nullable = true)
 |-- trip_distance_km: double (nullable = true)
 |-- trip_speed_kmh: double (nullable = true)
 |-- period_of_day: string (nullable = true)
 |-- start_month: integer (nullable = true)
 |-- age_group: string (nullable = true)
 |-- season: string (nullab

f) Over-utilized bikes needing maintenance


In [ ]:
# Aggregate total trip seconds and trip count per bike
bike_usage_df = clean_df.groupBy("bikeid").agg(
    F.count("*").alias("total_trips"),
    F.sum("trip_duration_sec").alias("total_seconds"),
    F.round(F.avg("trip_duration_sec"), 2).alias("avg_trip_sec")
)

In [ ]:
# Convert total seconds to hours
bike_usage_df = bike_usage_df.withColumn(
    "total_hours",
    F.round(F.col("total_seconds") / 3600, 2)
)

In [ ]:
# flag bikes that exceed the 90th-percentile threshold of total seconds
# use here approxQuantile so we don't need to collect the whole dataset
threshold_90 = clean_df.approxQuantile("trip_duration_sec", [0.90], 0.01)[0]
total_sec_90th = bike_usage_df.approxQuantile("total_seconds", [0.90], 0.01)[0]

bike_usage_df=bike_usage_df.withColumn(  "needs_maintenance", F.when(F.col("total_seconds") >= total_sec_90th, "Yes").otherwise("No"))

In [ ]:
print("top 20 used bikes")
bike_usage_df.orderBy(F.desc("total_seconds")).show(20, truncate=False)

print("Bike count needing maintenance")
bike_usage_df.groupBy("needs_maintenance").count().show()

print("maintainance percentage")
bike_usage_df.groupBy("needs_maintenance").count().withColumn(
    "percentage",F.round(F.col("count") / bike_usage_df.count() * 100, 2)).show()


top 20 used bikes
+------+-----------+-------------+------------+-----------+-----------------+
|bikeid|total_trips|total_seconds|avg_trip_sec|total_hours|needs_maintenance|
+------+-----------+-------------+------------+-----------+-----------------+
|40515 |29         |2964195      |102213.62   |823.39     |Yes              |
|25073 |47         |2632367      |56007.81    |731.21     |Yes              |
|32225 |100        |2457156      |24571.56    |682.54     |Yes              |
|21580 |56         |2313519      |41312.84    |642.64     |Yes              |
|26495 |66         |2210383      |33490.65    |614.0      |Yes              |
|15876 |98         |2194728      |22395.18    |609.65     |Yes              |
|21541 |62         |2075319      |33472.89    |576.48     |Yes              |
|27447 |91         |2021581      |22215.18    |561.55     |Yes              |
|17509 |47         |1935308      |41176.77    |537.59     |Yes              |
|32295 |82         |1811080      |22086.34    

# Interpretation for f:

Some bikes have more total ride time than others, making them prime candidates fo inspections. Only about 11% of bikes need maintenance

# How this affects decision making:

Schedule maintenance for the flagged 11% of bikes before peak hours to avoid mid-ride failures.

# Additional observation:

bike 40515 has only 29 trips but 823 hours of total ride time, meaning individual trips were extremely long (avg 28 hours each). This suggests the bike was maybe stolen .


Decision: Flag bikes with very high avg_trip_sec for a different kind of investigation (theft/malfunction) separate from the standard maintenance check.

g) Subscriber vs customer destination analysis


In [ ]:
# Count trips to each end station, split by user type
dest_df = clean_df.groupBy("usertype", "end station name").agg(
    F.count("*").alias("trip_count")
)

# Rank destinations within each user type using a window function
window_spec = Window.partitionBy("usertype").orderBy(F.desc("trip_count"))
dest_ranked = dest_df.withColumn("rank", F.rank().over(window_spec))

#Keep only the top 10 destinations per user type
top_dest = dest_ranked.filter(F.col("rank") <= 10).orderBy("usertype", "rank")

print(" top 10 subscribers ")
top_dest.filter(F.col("usertype") == "Subscriber") \
        .select("rank", "end station name", "trip_count").show(10, truncate=False)

print("top 10 customers ")
top_dest.filter(F.col("usertype") == "Customer") \
        .select("rank", "end station name", "trip_count").show(10, truncate=False)


 top 10 subscribers 
+----+-----------------------------+----------+
|rank|end station name             |trip_count|
+----+-----------------------------+----------+
|1   |Pershing Square North        |9263      |
|2   |Broadway & E 22 St           |7372      |
|3   |E 17 St & Broadway           |7113      |
|4   |8 Ave & W 31 St              |7053      |
|5   |W 21 St & 6 Ave              |6727      |
|6   |Broadway & E 14 St           |6478      |
|7   |Lafayette St & E 8 St        |5833      |
|8   |West St & Chambers St        |5753      |
|9   |Christopher St & Greenwich St|5673      |
|10  |E 47 St & Park Ave           |5577      |
+----+-----------------------------+----------+

top 10 customers 
+----+---------------------------------+----------+
|rank|end station name                 |trip_count|
+----+---------------------------------+----------+
|1   |5 Ave & E 88 St                  |2363      |
|2   |Central Park S & 6 Ave           |2336      |
|3   |5 Ave & E 73 St       

# Business interpreation of G:
Subscribers end trips at business locations(Pershing Square, Penn Station area), meaning subscibers are working class members riding to work areas. Customers ride often around Central Park and tourist landmarks, meaning customers are more often tourists.

# How this can support decision making:
Stock more bikes at business hubs on weekday mornings and at tourist spots on weekends.

h) Most demanded station pairs


In [ ]:
# group by the start/end station pair and count trips
pairs_df = clean_df.groupBy("start station name", "end station name").agg(
    F.count("*").alias("trip_count"),
    F.round(F.avg("trip_duration_sec"), 2).alias("avg_duration_sec"),
    F.round(F.avg("trip_distance_km"), 3).alias("avg_distance_km")
)

# exclude trivial round trips (same start and end station) and focus on different pickup-dropoff rides
pairs_df = pairs_df.filter(
    F.col("start station name") != F.col("end station name")
)

print("Top 15 Most Demanded Station Pairs")
pairs_df.orderBy(F.desc("trip_count")).show(15, truncate=False)


Top 15 Most Demanded Station Pairs
+-----------------------------+-----------------------------+----------+----------------+---------------+
|start station name           |end station name             |trip_count|avg_duration_sec|avg_distance_km|
+-----------------------------+-----------------------------+----------+----------------+---------------+
|E 7 St & Avenue A            |Cooper Square & Astor Pl     |585       |240.82          |0.691          |
|Central Park S & 6 Ave       |5 Ave & E 88 St              |443       |1740.99         |2.383          |
|Yankee Ferry Terminal        |Soissons Landing             |411       |3690.9          |0.624          |
|Vesey Pl & River Terrace     |North Moore St & Greenwich St|401       |339.21          |0.756          |
|Soissons Landing             |Picnic Point                 |379       |2043.2          |1.192          |
|Soissons Landing             |Yankee Ferry Terminal        |371       |1673.73         |0.624          |
|Picnic Poi

# Business Interpretation of H:


The busiest corridors are short, frequent commuter routes (under 1 km), while pairs like Central Park S , 5 Ave and Soissons Landing Picnic Point show longer  duration ride time.

# Decision making influence:
 These high-demand pairs should be prioritized for bike availability and rebalancing to prevent bike shortages on high-demand routes.

i) Gender-based riding behavior differences


In [ ]:
# map numeric gender codes to gender labels
gender_labels = clean_df.withColumn(
    "gender_label",
    F.when(F.col("gender") == 1, "Male")
     .when(F.col("gender") == 2, "Female")
     .otherwise("Unknown")
)

gender_labels.groupBy("gender", "gender_label").count().orderBy("gender").show()

+------+------------+------+
|gender|gender_label| count|
+------+------------+------+
|     0|     Unknown|101378|
|     1|        Male|885821|
|     2|      Female|312113|
+------+------------+------+



In [ ]:
# Ratio of male to female riders
gender_counts = gender_labels.groupBy("gender_label").count()

male_count = gender_counts.filter(F.col("gender_label") == "Male").collect()[0]["count"]
female_count = gender_counts.filter(F.col("gender_label") == "Female").collect()[0]["count"]

ratio = round(male_count / female_count, 2)
print(f"Male : Female ratio = {ratio} : 1")
print(f"Males: {male_count:,} | Females: {female_count:,}")

Male : Female ratio = 2.84 : 1
Males: 885,821 | Females: 312,113


In [ ]:
# aggregate key riding metrics per gender group
result_i= gender_labels.groupBy("gender_label").agg(
    F.count("*").alias("trip_count"),
    F.round(F.avg("trip_duration_sec"), 2).alias("avg_duration_sec"),
    F.round(F.avg("trip_speed_kmh"), 3).alias("avg_speed_kmh"),
    F.round(F.avg("trip_distance_km"), 3).alias("avg_distance_km"),
    F.round(F.stddev("trip_duration_sec"), 2).alias("stddev_duration_sec"),
    F.round(F.stddev("trip_speed_kmh"), 3).alias("stddev_speed_kmh")
).orderBy("gender_label")

print("riding Behavior by Gender")
result_i.show(truncate=False)

riding Behavior by Gender
+------------+----------+----------------+-------------+---------------+-------------------+----------------+
|gender_label|trip_count|avg_duration_sec|avg_speed_kmh|avg_distance_km|stddev_duration_sec|stddev_speed_kmh|
+------------+----------+----------------+-------------+---------------+-------------------+----------------+
|Female      |312113    |988.07          |8.261        |1.83           |7651.02            |3.03            |
|Male        |885821    |831.77          |9.216        |1.733          |7379.2             |3.318           |
|Unknown     |101378    |1998.04         |6.421        |1.973          |22465.36           |3.556           |
+------------+----------+----------------+-------------+---------------+-------------------+----------------+



In [ ]:
# statistical comparison — coefficient of variation (CV)
print("Coefficient of Variation (Duration)")
result_i.select(
    "gender_label",
    "avg_duration_sec",
    "stddev_duration_sec",
    F.round(F.col("stddev_duration_sec") / F.col("avg_duration_sec"), 4)
     .alias("cv_duration")
).show(truncate=False)

#CV (Coefficient of Variation) measures how consistent a group's behavior is .The lower the value, the more predictable the rides.

Coefficient of Variation (Duration)
+------------+----------------+-------------------+-----------+
|gender_label|avg_duration_sec|stddev_duration_sec|cv_duration|
+------------+----------------+-------------------+-----------+
|Female      |988.07          |7651.02            |7.7434     |
|Male        |831.77          |7379.2             |8.8717     |
|Unknown     |1998.04         |22465.36           |11.2437    |
+------------+----------------+-------------------+-----------+



**Business Interpretation of i:**

Males outnumber females by a ratio of 2.84:1 (885,821 males vs 312,113 females), suggesting the system is used more by male commuters. Female riders take longer trips on average (988 seconds vs 831 seconds for males) and ride slower (8.26 km/h vs 9.22 km/h). Unknown gender riders (casual users) show the most unpredictable behavior with the highest CV of 11.24, indicating no consistent riding pattern.

Decision making influence:
Since males dominate usage nearly 3:1, target campaigns toward female riders to increase the proportion of female riders.

j) Weekday vs weekend behavior analysis


In [ ]:
#classify each trip as Weekday or Weekend . note:  1=Sunday, 7=Saturday
day_df = clean_df.withColumn(
    "day_type",
    F.when(F.dayofweek("starttime").isin(1, 7), "Weekend").otherwise("Weekday")
)

# Compare average duration, distance, and speed between the two groups
result_j = day_df.groupBy("day_type").agg(
    F.count("*").alias("trip_count"),
    F.round(F.avg("trip_duration_sec"), 2).alias("avg_duration_sec"),
    F.round(F.avg("trip_distance_km"), 3).alias("avg_distance_km"),
    F.round(F.avg("trip_speed_kmh"), 3).alias("avg_speed_kmh")
).orderBy("day_type")

result_j.show(truncate=False)

+--------+----------+----------------+---------------+-------------+
|day_type|trip_count|avg_duration_sec|avg_distance_km|avg_speed_kmh|
+--------+----------+----------------+---------------+-------------+
|Weekday |973745    |892.57          |1.763          |9.052        |
|Weekend |325567    |1162.93         |1.81           |7.92         |
+--------+----------+----------------+---------------+-------------+



**Business Interpretation (j):**

Weekday trips (973,745) outnumber weekend trips (325,567) by nearly 3:1. Weekend riders take longer trips on average (1163 seconds vs 893 seconds), travel slightly longer distances (1.81 km vs 1.76 km), and ride slower (7.92 km/h vs 9.05 km/h). This confirms that weekday trips are dominated by faster, shorter commutes, while weekend trips are longer, slower leisure rides.

Decision‑making influence: Prioritize bike availability on weekdays due to higher demand, but ensure sufficient bikes in tourist/recreational areas on weekends.

In [ ]:
#for dashboard

bike_usage_df.orderBy(F.desc("total_seconds")).limit(50) \
    .write.mode("overwrite").parquet("query_f_bike_utilization.parquet")

top_dest.write.mode("overwrite").parquet("query_g_destinations.parquet")

pairs_df.orderBy(F.desc("trip_count")).limit(30) \
    .write.mode("overwrite").parquet("query_h_station_pairs.parquet")

result_i.write.mode("overwrite").parquet("query_i_gender_behavior.parquet")

result_j.write.mode("overwrite").parquet("query_j_weekday_weekend.parquet")

# Member 5

In [ ]:
#Cell 1: Imports & Load Dataset
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler
)

from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier
)

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

from pyspark.sql import functions as F

# Load feature engineered dataset from Member 2
feature_df = spark.read.parquet("feature_engineered_citi_data.parquet")

# Remove noisy rows and unknown gender for ML prediction
# Project requires predicting 'gender', so filter out 'gender == 0' (Unknown)
ml_df = feature_df.filter(
    (F.col("noise_flag") == 0) &
    (F.col("speed_noise_flag") == 0) &
    (F.col("gender") != 0) # Filter out unknown gender for prediction
)

print(f"Rows used for ML: {ml_df.count():,}")
ml_df.printSchema()

Rows used for ML: 1,197,934
root
 |-- starttime: timestamp (nullable = true)
 |-- stoptime: timestamp (nullable = true)
 |-- start station name: string (nullable = true)
 |-- start station latitude: double (nullable = true)
 |-- start station longitude: double (nullable = true)
 |-- end station name: string (nullable = true)
 |-- end station latitude: double (nullable = true)
 |-- end station longitude: double (nullable = true)
 |-- bikeid: integer (nullable = true)
 |-- usertype: string (nullable = true)
 |-- birth year: integer (nullable = true)
 |-- gender: integer (nullable = true)
 |-- noise_flag: integer (nullable = true)
 |-- rider_age: integer (nullable = true)
 |-- trip_duration_sec: long (nullable = true)
 |-- trip_distance_km: double (nullable = true)
 |-- trip_speed_kmh: double (nullable = true)
 |-- period_of_day: string (nullable = true)
 |-- start_month: integer (nullable = true)
 |-- age_group: string (nullable = true)
 |-- season: string (nullable = true)
 |-- speed_no

In [ ]:
#Cell 2: Select ML Features
# Selected features
categorical_cols = [
    # "gender", # Removed: 'gender' is the target label, not a feature
    "period_of_day",
    "age_group",
    "season"
]

numeric_cols = [
    "trip_duration_sec",
    "trip_distance_km",
    "trip_speed_kmh",
    "rider_age"
]

# Keep only needed columns
ml_df = ml_df.select(
    categorical_cols +
    numeric_cols +
    ["gender", "usertype"] # Keep 'gender' as it will be transformed into 'label'
)

In [ ]:
#Cell 3: Handle Missing Values
# Remove rows with null values in important columns
ml_df = ml_df.dropna()

print(f"Rows after removing NULLs: {ml_df.count():,}")

Rows after removing NULLs: 1,197,934


In [ ]:
#Cell 4: Encode Labels & Categorical Features
# Encode target label
label_indexer = StringIndexer(
    inputCol="gender", # Changed from "usertype" to "gender"
    outputCol="label"
)

# Encode categorical columns
indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=col + "_index",
        handleInvalid="keep"
    )
    for col in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=col + "_index",
        outputCol=col + "_vec"
    )
    for col in categorical_cols
]

In [ ]:
#Cell 5: Assemble Features
# Final encoded categorical vectors
encoded_features = [col + "_vec" for col in categorical_cols]

# Combine all features
assembler = VectorAssembler(
    inputCols=encoded_features + numeric_cols,
    outputCol="assembled_features"
)

In [ ]:
#Cell 6: Normalize Numerical Features
# Normalize features
scaler = StandardScaler(
    inputCol="assembled_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

In [ ]:
#Cell 7: Train/Test Split
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows: {train_df.count():,}")
print(f"Testing rows : {test_df.count():,}")

Training rows: 958,420
Testing rows : 239,514


In [ ]:
#Cell 8: Logistic Regression Model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20
)

lr_pipeline = Pipeline(stages=
    [label_indexer] +
    indexers +
    encoders +
    [assembler, scaler, lr]
)

lr_model = lr_pipeline.fit(train_df)

lr_predictions = lr_model.transform(test_df)

lr_predictions.select(
    "gender", # Changed usertype to gender for viewing actual vs predicted
    "prediction",
    "probability"
).show(5, truncate=False)

+------+----------+----------------------------------------+
|gender|prediction|probability                             |
+------+----------+----------------------------------------+
|1     |0.0       |[0.6182485890232415,0.38175141097675847]|
|2     |0.0       |[0.5639248612530815,0.4360751387469185] |
|1     |0.0       |[0.5913661168486868,0.4086338831513132] |
|1     |0.0       |[0.8443945974455217,0.1556054025544783] |
|1     |0.0       |[0.915948694443134,0.08405130555686602] |
+------+----------+----------------------------------------+
only showing top 5 rows


In [ ]:
#Cell 9: Decision Tree Model
dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=5
)

dt_pipeline = Pipeline(stages=
    [label_indexer] +
    indexers +
    encoders +
    [assembler, scaler, dt]
)

dt_model = dt_pipeline.fit(train_df)

dt_predictions = dt_model.transform(test_df)

In [ ]:
#Cell 10: Random Forest Model
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=20,
    maxDepth=7,
    seed=42
)

rf_pipeline = Pipeline(stages=
    [label_indexer] +
    indexers +
    encoders +
    [assembler, scaler, rf]
)

rf_model = rf_pipeline.fit(train_df)

rf_predictions = rf_model.transform(test_df)

In [ ]:
#Cell 11: Evaluate Model Accuracy
# Accuracy evaluator
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

# Logistic Regression accuracy
lr_accuracy = evaluator.evaluate(lr_predictions)

# Decision Tree accuracy
dt_accuracy = evaluator.evaluate(dt_predictions)

# Random Forest accuracy
rf_accuracy = evaluator.evaluate(rf_predictions)

print("=== Model Accuracy Comparison ===")
print(f"Logistic Regression Accuracy : {lr_accuracy:.4f}")
print(f"Decision Tree Accuracy       : {dt_accuracy:.4f}")
print(f"Random Forest Accuracy       : {rf_accuracy:.4f}")

=== Model Accuracy Comparison ===
Logistic Regression Accuracy : 0.7379
Decision Tree Accuracy       : 0.7381
Random Forest Accuracy       : 0.7381


In [ ]:
#Cell 12: Accuracy Comparison Table
comparison_df = spark.createDataFrame([
    ("Logistic Regression", float(lr_accuracy)),
    ("Decision Tree", float(dt_accuracy)),
    ("Random Forest", float(rf_accuracy))
], ["Model", "Accuracy"])

comparison_df.orderBy(F.desc("Accuracy")).show(truncate=False)

+-------------------+------------------+
|Model              |Accuracy          |
+-------------------+------------------+
|Decision Tree      |0.7381405679834999|
|Random Forest      |0.7381405679834999|
|Logistic Regression|0.7378566597359654|
+-------------------+------------------+



In [ ]:
import numpy as np
#Cell 13: Feature Importance (Random Forest)
# Extract trained RF model from pipeline
rf_stage_model = rf_model.stages[-1]

# Feature importances
importance_values = rf_stage_model.featureImportances

# Feature names - IMPORTANT: Re-define feature_names to reflect the current features
# The order of features in VectorAssembler is: encoded_features + numeric_cols
# where encoded_features are from categorical_cols and numeric_cols are directly used
feature_names = [col + "_vec" for col in categorical_cols] + numeric_cols

# Convert numpy.float64 to native Python float before creating DataFrame
importance_data = list(zip(feature_names, [float(x) for x in importance_values]))

importance_df = spark.createDataFrame(
    importance_data,
    ["Feature", "Importance"]
)

print("=== Feature Importance ===")
importance_df.orderBy(F.desc("Importance")).show(truncate=False)

=== Feature Importance ===
+-----------------+---------------------+
|Feature          |Importance           |
+-----------------+---------------------+
|trip_duration_sec|0.05152594876376928  |
|trip_speed_kmh   |0.020792415753596576 |
|period_of_day_vec|0.0013566781537533064|
|rider_age        |7.95063427009295E-4  |
|season_vec       |5.666850108642473E-4 |
|trip_distance_km |1.0176298661102585E-4|
|age_group_vec    |0.0                  |
+-----------------+---------------------+



In [ ]:
#Cell 14: Save Results for Dashboard
comparison_df.write.mode("overwrite") \
    .parquet("ml_model_comparison.parquet")

importance_df.write.mode("overwrite") \
    .parquet("ml_feature_importance.parquet")

print("Saved ML outputs successfully ✓")

Saved ML outputs successfully ✓


# Model Evaluation Discussion

This section details the performance of the three machine learning models trained to predict rider gender, along with an analysis of their strengths, weaknesses, and key feature importances.

---

## 1. Logistic Regression

**Strengths:**
*   **Fast to train:** Efficient for large datasets.
*   **Works well on linear relationships:** Interpretable when features have a clear linear correlation with the outcome.
*   **Easy to interpret:** The coefficients provide insights into feature impact.

**Weaknesses:**
*   **Cannot capture complex non-linear patterns:** Performance degrades with intricate data relationships.
*   **Lower flexibility:** Less adaptable than tree-based models for complex datasets.

---

## 2. Decision Tree

**Strengths:**
*   **Easy to visualize:** The decision-making process can be clearly represented.
*   **Handles non-linear relationships:** Can model complex interactions between features.
*   **Works well with mixed feature types:** Accommodates both numerical and categorical data naturally.

**Weaknesses:**
*   **Can overfit easily:** Prone to learning noise in the training data, leading to poor generalization.
*   **Less stable:** Small changes in data can lead to significantly different tree structures.

---

## 3. Random Forest

**Strengths:**
*   **Highest accuracy in most cases:** Often provides superior predictive performance.
*   **Reduces overfitting:** Achieves this by averaging the predictions of multiple trees, each trained on a subset of the data.
*   **Handles complex relationships well:** Robust for datasets with intricate patterns.
*   **More robust and stable:** Less sensitive to individual data point changes compared to single decision trees.

**Weaknesses:**
*   **Slower than a single decision tree:** Training can be more computationally intensive.
*   **Harder to interpret:** The ensemble nature makes it difficult to visualize a single decision path.

---

## Model Performance Summary

Random Forest and Decision Tree achieved **identical accuracy (73.81%)**, slightly outperforming Logistic Regression (73.79%). While the accuracy is similar, Random Forest offers better generalization and robustness against overfitting, making it the preferred model for this task.

Random Forest's ensemble approach combines predictions from numerous decision trees. This architecture leads to improved results because:

*   **Diversity in Learning:** Different trees learn different patterns from various subsets of the data and features, capturing a wider range of relationships.
*   **Reduction of Overfitting:** By aggregating predictions, the model mitigates the risk of overfitting that individual decision trees often exhibit.
*   **Better Generalization:** The combined wisdom of multiple trees leads to more stable and accurate predictions on unseen data.
*   **Robustness to Noise:** The ensemble method inherently makes it more resistant to noise and outliers in the dataset.

For the Citi Bike data, rider behavior is inherently **complex and non-linear**. Factors like `trip_duration_sec`, `trip_speed_kmh`, `season`, `period_of_day`, and `rider_age` interact in multifaceted ways to influence gender. Random Forest is adept at identifying and leveraging these complex patterns.

## Most Influential Features

From the Random Forest feature importance output:

- **`trip_duration_sec`** is the most influential feature (importance ~0.0515).  
- **`trip_speed_kmh`** is the second most important (~0.0208).  
- **`period_of_day_vec`** (encoded time of day) has moderate importance (~0.00136).  
- **`rider_age`**, **`season_vec`**, and **`trip_distance_km`** have very low importance.  
- **`age_group_vec`** contributed zero importance, suggesting that the broader age categories (Young/Adult/Senior) do not help predict gender.

This indicates that precise trip duration and speed are far more predictive of rider gender than demographic or seasonal factors.

member 6 — Dashboard, Documentation & Final Integration


In [ ]:
!pip install bokeh pandas -q

In [ ]:
# ── Member 6 | Final Dashboard (Loads Parquet Files from Members 3-5) ──────────
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, HoverTool, Select, CustomJS
from bokeh.layouts import column, row
import pandas as pd

output_notebook()

# ------------------------------
# Load query results (parquet files)
# ------------------------------
df_hourly = spark.read.parquet("query_c_hourly.parquet").toPandas().sort_values("hour_of_day")
df_top_stations = spark.read.parquet("query_b_top_stations.parquet").toPandas().head(10)
df_age = spark.read.parquet("query_d_agegroup.parquet").toPandas()
df_gender = spark.read.parquet("query_i_gender_behavior.parquet").toPandas()
df_season = spark.read.parquet("query_e_seasonal.parquet").toPandas()
df_bike = spark.read.parquet("query_f_bike_utilization.parquet").toPandas().head(10)
df_dest = spark.read.parquet("query_g_destinations.parquet").toPandas()

# Maintenance percentage (from Member 4 output, not saved as parquet – we use hardcoded numbers)
maintenance_data = {"status": ["No", "Yes"], "percentage": [89.18, 10.82]}
df_maintenance = pd.DataFrame(maintenance_data)

# ------------------------------
# 1. Hourly Demand
# ------------------------------
p1 = figure(title="Hourly Trip Demand", x_axis_label="Hour", y_axis_label="Trips", width=600, height=400)
p1.line(df_hourly["hour_of_day"], df_hourly["trip_count"], line_width=2, color="navy")
p1.scatter(df_hourly["hour_of_day"], df_hourly["trip_count"], size=4, color="red")
p1.add_tools(HoverTool(tooltips=[("Hour", "@x"), ("Trips", "@y")]))

# ------------------------------
# 2. Top 10 Start Stations
# ------------------------------
df_stations = df_top_stations.sort_values("trip_count", ascending=True)
source_stations = ColumnDataSource(df_stations)
p2 = figure(title="Top 10 Start Stations", y_range=list(df_stations["start station name"]),
            x_axis_label="Trips", width=600, height=400)
p2.hbar(y="start station name", right="trip_count", source=source_stations, height=0.8, color="orange")
p2.add_tools(HoverTool(tooltips=[("Station", "@{start station name}"), ("Trips", "@trip_count")]))

# ------------------------------
# 3. Trip Duration by Age Group
# ------------------------------
source_age = ColumnDataSource(df_age)
p3 = figure(title="Avg Trip Duration by Age Group", x_range=df_age["age_group"],
            x_axis_label="Age Group", y_axis_label="Seconds", width=500, height=400)
p3.vbar(x="age_group", top="avg_duration_sec", source=source_age, width=0.6, color="green")
p3.add_tools(HoverTool(tooltips=[("Age Group", "@age_group"), ("Avg Duration", "@avg_duration_sec")]))

# ------------------------------
# 4. Gender Speed Comparison
# ------------------------------
source_gender = ColumnDataSource(df_gender)
p4 = figure(title="Avg Speed by Gender", x_range=df_gender["gender_label"],
            x_axis_label="Gender", y_axis_label="km/h", width=500, height=400)
p4.vbar(x="gender_label", top="avg_speed_kmh", source=source_gender, width=0.6, color="purple")
p4.add_tools(HoverTool(tooltips=[("Gender", "@gender_label"), ("Speed", "@avg_speed_kmh")]))

# ------------------------------
# 5. Seasonal Trends
# ------------------------------
source_season = ColumnDataSource(df_season)
p5 = figure(title="Seasonal Trip Distribution", x_range=df_season["season"],
            x_axis_label="Season", y_axis_label="Trips", width=400, height=400)
p5.vbar(x="season", top="trip_count", source=source_season, width=0.6, color="lightblue")
p5.add_tools(HoverTool(tooltips=[("Season", "@season"), ("Trips", "@trip_count")]))

# ------------------------------
# 6. Bike Utilization (Top 10 by total hours) - FIXED: convert bikeid to string
# ------------------------------
df_bike_sorted = df_bike.sort_values("total_hours", ascending=True)
df_bike_sorted["bikeid_str"] = df_bike_sorted["bikeid"].astype(str)
source_bike = ColumnDataSource(df_bike_sorted)
p6 = figure(title="Top 10 Bikes by Total Hours", y_range=list(df_bike_sorted["bikeid_str"]),
            x_axis_label="Hours", width=600, height=400)
p6.hbar(y="bikeid_str", right="total_hours", source=source_bike, height=0.8, color="brown")
p6.add_tools(HoverTool(tooltips=[("Bike", "@bikeid_str"), ("Hours", "@total_hours")]))

# ------------------------------
# 7. Maintenance Percentage
# ------------------------------
source_maint = ColumnDataSource(df_maintenance)
p7 = figure(title="Bikes Needing Maintenance", x_range=df_maintenance["status"],
            x_axis_label="Maintenance", y_axis_label="%", width=400, height=400)
p7.vbar(x="status", top="percentage", source=source_maint, width=0.6, color="darkred")
p7.add_tools(HoverTool(tooltips=[("Status", "@status"), ("Percent", "@percentage")]))

# ------------------------------
# 8. Interactive Filter: Top End Stations by User Type
# ------------------------------
sub_df = df_dest[df_dest["usertype"] == "Subscriber"].sort_values("trip_count", ascending=True)
cust_df = df_dest[df_dest["usertype"] == "Customer"].sort_values("trip_count", ascending=True)

sub_stations = sub_df["end station name"].tolist()
sub_trips = sub_df["trip_count"].tolist()
cust_stations = cust_df["end station name"].tolist()
cust_trips = cust_df["trip_count"].tolist()

source_filter = ColumnDataSource(data=dict(station=sub_stations, trips=sub_trips))
p_filter = figure(title="Top End Stations by User Type (Interactive)", y_range=sub_stations,
                  x_axis_label="Number of Trips", width=600, height=400)
p_filter.hbar(y="station", right="trips", source=source_filter, height=0.8, color="brown")
p_filter.add_tools(HoverTool(tooltips=[("Station", "@station"), ("Trips", "@trips")]))

select = Select(title="User Type:", value="Subscriber", options=["Subscriber", "Customer"])
callback = CustomJS(args=dict(source=source_filter,
                              sub_stations=sub_stations, sub_trips=sub_trips,
                              cust_stations=cust_stations, cust_trips=cust_trips,
                              p=p_filter), code="""
    var type = cb_obj.value;
    var new_stations, new_trips;
    if (type === "Subscriber") {
        new_stations = sub_stations;
        new_trips = sub_trips;
    } else {
        new_stations = cust_stations;
        new_trips = cust_trips;
    }
    source.data = { station: new_stations, trips: new_trips };
    p.y_range.factors = new_stations;
    source.change.emit();
""")
select.js_on_change("value", callback)
filter_layout = column(select, p_filter)

# ------------------------------
# Show all visuals
# ------------------------------
show(row(p1, p2))
show(row(p3, p4))
show(row(p5, p6))
show(p7)
show(filter_layout)